In [154]:
import pandas as pd
import numpy as np
from pathlib import Path

In [155]:
cwd = Path.cwd()
trade_file = cwd/ "trade_data_example.csv"
df = pd.read_csv(trade_file)

market_file = cwd/ "market_price_example.csv"
df_market = pd.read_csv(market_file)

In [156]:
df

,ID,Book,Commodity,Delivery,Action,Qty,Price(£/MWh)
0,TR-001,Physical,NBPGas,January,Buy,500,25.0
1,TR-002,Physical,NBPGas,January,Sell,300,26.0
2,TR-003,Hedge,NBPGas,January,Sell,150,25.5
3,TR-004,Physical,NBPGas,February,Buy,400,24.0
4,TR-005,Physical,NBPGas,February,Sell,320,25.0
5,TR-006,Hedge,NBPGas,February,Sell,60,24.5


In [157]:
df_market

,Commodity,Delivery,Market Price(£/MWh)
0,NBPGas,January,27
1,NBPGas,February,26


In [164]:
# Calculate signed quantities
def trades(df):
    # df.assign(Signed=np.where(df['Action']=='Buy', df['Qty'], df['Qty']*(-1)))
    # df.style.format({'Signed':"{0:+g}"})

    # Source - https://stackoverflow.com/a/73664277
    # Posted by cottontail
    # Retrieved 2026-10-06, License - CC BY-SA 4.0

    df['SignedQty'] = pd.Series(df['Qty']*(-1), index=df.index).mask(df['Action']=='Buy', df['Qty'])
    # df['SignedQty'] = df['SignedQty'].apply(lambda x: f"+{x}" if x > 0 else f"{x}")
    return df

# Calculate Aggregrate positions
def positions(df):
    return trades(df).groupby(['Commodity', 'Delivery', 'Book'])['SignedQty'].sum().reset_index().rename(columns={'SignedQty': 'Net Position'})

# Calculate internal risk metrics: Delta exposure and financial hedge coverage
def exposure(df):
    position = positions(df)
    exposure = position.pivot_table(
        index = ['Commodity', 'Delivery'],
        columns = 'Book',
        values = 'Net Position',
        fill_value = 0
    ).reset_index()
    exposure['Delta'] = exposure['Physical'] + exposure['Hedge']
    exposure['Coverage %'] = abs(exposure['Hedge'])/abs(exposure['Physical'])*100
    exposure =  exposure.rename_axis(None, axis=1).reset_index(drop=True)

    print(f"Total unhedged exposure {exposure['Delta'].sum()} MWh")
    exposure.to_csv("exposure_example.csv")
    return exposure

# Join market prices to trades
def trades_market(df, df_market):
    return trades(df).merge(df_market, on=['Commodity', 'Delivery'], how='left')

# Calculate mark-to-market P&L
def MtM_PnL(df, df_market):
    trades = trades_market(df, df_market)
    trades['MtM_PnL(£)'] = (trades['Market Price(£/MWh)'] - trades['Price(£/MWh)']) * trades['SignedQty']
    print(trades.groupby('Book')['MtM_PnL(£)'].sum().reset_index().rename_axis(None, axis=1))
    print(f"Total P&L : {trades['MtM_PnL(£)'].sum()} £")
    trades.to_csv("PnL_report_example.csv")
    return trades

In [159]:
trades(df)

,ID,Book,Commodity,Delivery,Action,Qty,Price(£/MWh),SignedQty
0,TR-001,Physical,NBPGas,January,Buy,500,25.0,500
1,TR-002,Physical,NBPGas,January,Sell,300,26.0,-300
2,TR-003,Hedge,NBPGas,January,Sell,150,25.5,-150
3,TR-004,Physical,NBPGas,February,Buy,400,24.0,400
4,TR-005,Physical,NBPGas,February,Sell,320,25.0,-320
5,TR-006,Hedge,NBPGas,February,Sell,60,24.5,-60


In [160]:
positions(df)

,Commodity,Delivery,Book,Net Position
0,NBPGas,February,Hedge,-60
1,NBPGas,February,Physical,80
2,NBPGas,January,Hedge,-150
3,NBPGas,January,Physical,200


In [161]:
exposure(df)

Total unhedged exposure 70.0 MWh


,Commodity,Delivery,Hedge,Physical,Delta,Coverage %
0,NBPGas,February,-60.0,80.0,20.0,75.0
1,NBPGas,January,-150.0,200.0,50.0,75.0


In [162]:
trades_market(df, df_market)

,ID,Book,Commodity,Delivery,Action,Qty,Price(£/MWh),SignedQty,Market Price(£/MWh)
0,TR-001,Physical,NBPGas,January,Buy,500,25.0,500,27
1,TR-002,Physical,NBPGas,January,Sell,300,26.0,-300,27
2,TR-003,Hedge,NBPGas,January,Sell,150,25.5,-150,27
3,TR-004,Physical,NBPGas,February,Buy,400,24.0,400,26
4,TR-005,Physical,NBPGas,February,Sell,320,25.0,-320,26
5,TR-006,Hedge,NBPGas,February,Sell,60,24.5,-60,26


In [165]:
MtM_PnL(df, df_market)

       Book  MtM_PnL(£)
0     Hedge      -315.0
1  Physical      1180.0
Total P&L : 865.0 £


,ID,Book,Commodity,Delivery,Action,Qty,Price(£/MWh),SignedQty,Market Price(£/MWh),MtM_PnL(£)
0,TR-001,Physical,NBPGas,January,Buy,500,25.0,500,27,1000.0
1,TR-002,Physical,NBPGas,January,Sell,300,26.0,-300,27,-300.0
2,TR-003,Hedge,NBPGas,January,Sell,150,25.5,-150,27,-225.0
3,TR-004,Physical,NBPGas,February,Buy,400,24.0,400,26,800.0
4,TR-005,Physical,NBPGas,February,Sell,320,25.0,-320,26,-320.0
5,TR-006,Hedge,NBPGas,February,Sell,60,24.5,-60,26,-90.0
